# Field Layout
Draws SAY-compliant markings for soccer field polygons in ArcGIS Online.

**Before running:** fill in the item IDs below. Leave `FIELD_IDS` empty to process all fields.

In [ ]:
# ── Cell 1: Configuration ────────────────────────────────────────────────
# Item ID of the input field polygons feature layer
FIELDS_ITEM_ID = ""  # TODO: set this

# Item IDs of the four output feature layers (created manually in ArcGIS Online)
LINES_ITEM_ID   = ""  # TODO: set this
CIRCLES_ITEM_ID = ""  # TODO: set this
MASKS_ITEM_ID   = ""  # TODO: set this
POINTS_ITEM_ID  = ""  # TODO: set this

# Optional: limit processing to specific field OBJECTIDs (empty list = all)
FIELD_IDS = []  # e.g. [1, 2, 5]

# Name of the Pitch Type attribute on the field polygon layer
PITCH_TYPE_FIELD = "Pitch Type"

print("Configuration loaded.")

In [ ]:
# ── Cell 2: Connect & Load ────────────────────────────────────────────────
from arcgis.gis import GIS

# Auto-detect environment: ArcGIS Online notebook vs local Jupyter
try:
    gis = GIS("home")
    print("Connected via ArcGIS Online session.")
except Exception as e:
    print(f"GIS('home') unavailable ({type(e).__name__}), connecting to ArcGIS Online...")
    gis = GIS("https://www.arcgis.com")
    print(f"Connected as {gis.properties.user.username}")

# Validate all item IDs are set
missing = [name for name, val in [
    ("FIELDS_ITEM_ID", FIELDS_ITEM_ID),
    ("LINES_ITEM_ID", LINES_ITEM_ID),
    ("CIRCLES_ITEM_ID", CIRCLES_ITEM_ID),
    ("MASKS_ITEM_ID", MASKS_ITEM_ID),
    ("POINTS_ITEM_ID", POINTS_ITEM_ID),
] if not val]
if missing:
    raise ValueError(f"Missing item IDs in Cell 1: {missing}")

# Load feature layers
def _load_item(item_id, label):
    item = gis.content.get(item_id)
    if item is None:
        raise ValueError(f"Could not access ArcGIS item '{label}' ({item_id}). "
                         "Check the item ID and your permissions.")
    return item

fields_item = _load_item(FIELDS_ITEM_ID, "FIELDS_ITEM_ID")
lines_fl    = _load_item(LINES_ITEM_ID,   "LINES_ITEM_ID").layers[0]
circles_fl  = _load_item(CIRCLES_ITEM_ID, "CIRCLES_ITEM_ID").layers[0]
masks_fl    = _load_item(MASKS_ITEM_ID,   "MASKS_ITEM_ID").layers[0]
points_fl   = _load_item(POINTS_ITEM_ID,  "POINTS_ITEM_ID").layers[0]
fields_fl   = fields_item.layers[0]

# Read fill color per pitch type from source layer renderer
pitch_colors = {}  # e.g. {"7v7": [0, 128, 0, 255], ...}
renderer = fields_fl.properties.get("drawingInfo", {}).get("renderer", {})
if renderer.get("type") == "uniqueValue":
    for info in renderer.get("uniqueValueInfos", []):
        value = info["value"]
        color = info.get("symbol", {}).get("color", [0, 0, 0, 0])
        pitch_colors[value] = color
    print(f"Extracted colors for pitch types: {list(pitch_colors.keys())}")
else:
    print("Warning: source layer renderer is not UniqueValue. Masks will use transparent fill.")

print("Layers loaded.")

In [ ]:
# ── Cell 3: Imports ───────────────────────────────────────────────────────
from geometry import yards_to_native
from field_markings import build_field_markings
from arcgis.geometry import Polyline, Polygon, Point  # used in Cell 5
from arcgis.features import Feature  # used in Cell 5
print("Imports OK.")

In [ ]:
# ── Cell 4: Read Fields ───────────────────────────────────────────────────
where = "1=1"
if FIELD_IDS:
    ids_str = ",".join(str(i) for i in FIELD_IDS)
    where = f"OBJECTID IN ({ids_str})"

field_features = fields_fl.query(
    where=where,
    out_fields=f"OBJECTID,{PITCH_TYPE_FIELD}",
    return_geometry=True,
).features

print(f"Loaded {len(field_features)} field(s).")

In [ ]:
# ── Cell 5: Compute Markings ──────────────────────────────────────────────
sr = fields_fl.properties.extent.spatialReference
scale = yards_to_native(sr)
sr_dict = {"wkid": sr["wkid"]}
print(f"Spatial reference: {sr}, scale = {scale:.4f} native units per yard")

lines_to_add    = []
circles_to_add  = []
masks_to_add    = []
points_to_add   = []
skipped         = []
processed_ids   = []

for feat in field_features:
    field_id   = feat.attributes.get("OBJECTID")
    pitch_type = feat.attributes.get(PITCH_TYPE_FIELD, "").strip()
    geom       = feat.geometry

    if not geom or not geom.get("rings"):
        skipped.append((field_id, "no geometry"))
        continue

    coords = [(pt[0], pt[1]) for pt in geom["rings"][0]]  # rings[0] is the exterior boundary
    if len(coords) < 3:
        skipped.append((field_id, "fewer than 3 vertices"))
        continue

    if not pitch_type:
        skipped.append((field_id, f"missing or blank {PITCH_TYPE_FIELD!r}"))
        continue

    try:
        markings = build_field_markings(coords, pitch_type, field_id, scale)
    except KeyError as e:
        skipped.append((field_id, str(e)))
        continue

    for f in markings["lines"]:
        g = {**f["geometry"], "spatialReference": sr_dict}
        lines_to_add.append(Feature(geometry=Polyline(g), attributes=f["attributes"]))

    for f in markings["circles"]:
        g = {**f["geometry"], "spatialReference": sr_dict}
        circles_to_add.append(Feature(geometry=Polygon(g), attributes=f["attributes"]))

    for f in markings["masks"]:
        g = {**f["geometry"], "spatialReference": sr_dict}
        masks_to_add.append(Feature(geometry=Polygon(g), attributes=f["attributes"]))

    for f in markings["points"]:
        g = {**f["geometry"], "spatialReference": sr_dict}
        points_to_add.append(Feature(geometry=Point(g), attributes=f["attributes"]))

    processed_ids.append(field_id)

print(f"Computed markings for {len(processed_ids)} field(s).")
if skipped:
    print(f"Skipped {len(skipped)} field(s):")
    for fid, reason in skipped:
        print(f"  OBJECTID {fid}: {reason}")

In [ ]:
# ── Cell 6: Clear Existing Markings (idempotent) ─────────────────────────
if not processed_ids:
    print("No fields to process — skipping clear.")
else:
    ids_str = ",".join(str(i) for i in processed_ids)
    where   = f"field_id IN ({ids_str})"
    for fl, name in [
        (lines_fl,   "Lines"),
        (circles_fl, "Circles"),
        (masks_fl,   "Masks"),
        (points_fl,  "Points"),
    ]:
        result = fl.delete_features(where=where)
        deleted = len(result.get("deleteResults", []))
        print(f"  {name}: deleted {deleted} existing feature(s).")

In [ ]:
# ── Cell 7: Write & Update Renderer ──────────────────────────────────────

def _add_features(fl, features, layer_name):
    if not features:
        print(f"  {layer_name}: nothing to add.")
        return
    result = fl.edit_features(adds=features)
    success = sum(1 for r in result.get("addResults", []) if r.get("success"))
    print(f"  {layer_name}: added {success}/{len(features)} feature(s).")

_add_features(lines_fl,   lines_to_add,   "Lines")
_add_features(circles_fl, circles_to_add, "Circles")
_add_features(masks_fl,   masks_to_add,   "Masks")
_add_features(points_fl,  points_to_add,  "Points")

# Update masks layer renderer — UniqueValueRenderer keyed on pitch_type
# using colors extracted from source layer in Cell 2
unique_value_infos = []
for pitch_type, color in pitch_colors.items():
    unique_value_infos.append({
        "value": pitch_type,
        "label": pitch_type,
        "symbol": {
            "type": "esriSFS",
            "style": "esriSFSSolid",
            "color": list(color),
            "outline": None,
        },
    })

if unique_value_infos:
    new_renderer = {
        "type": "uniqueValue",
        "field1": "pitch_type",
        "uniqueValueInfos": unique_value_infos,
        "defaultSymbol": {
            "type": "esriSFS",
            "style": "esriSFSSolid",
            "color": [0, 0, 0, 0],  # transparent fallback
            "outline": None,
        },
    }
    masks_fl.manager.update_definition({"drawingInfo": {"renderer": new_renderer}})
    print("Masks renderer updated.")
else:
    print("Warning: no pitch colors found — masks renderer not updated.")

print(f"\nDone. Processed {len(processed_ids)} field(s).")
if skipped:
    print(f"Skipped: {[f[0] for f in skipped]}")